In [ ]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import h5py

In [ ]:
def load_dataset():
    def load(path, prefix):
        f = h5py.File(path, 'r')
        x = np.array(f[f"{prefix}_set_x"][:])
        y = np.array(f[f"{prefix}_set_y"][:]).reshape(1, -1)
        return x, y

    train_x, train_y = load('datasets/catvnoncat/train_catvnoncat.h5', 'train')
    test_x,  test_y  = load('datasets/catvnoncat/test_catvnoncat.h5',  'test')
    classes = np.array(h5py.File('datasets/catvnoncat/test_catvnoncat.h5', 'r')["list_classes"][:])

    return train_x, train_y, test_x, test_y, classes

In [ ]:
m_train = train_set_x_orig.shape[0]
m_test = test_set_x_orig.shape[0]
num_px = train_set_x_orig.shape[1]

train_set_x = train_set_x_orig.reshape(train_set_x_orig.shape[0], -1).T / 255.
test_set_x  = test_set_x_orig.reshape(test_set_x_orig.shape[0], -1).T / 255.

In [ ]:
def sigmoid(z):
    s = 1 / (1 + np.exp(-z))
    return s

In [ ]:
def initialize(dim):
    w = np.zeros((dim, 1))
    b = 0.0
    return w, b

In [ ]:
def propagate(w, b, X, Y):
    m = X.shape[1]
    A = sigmoid(np.dot(w.T, X) + b)
    cost = -(1/m) * np.sum((Y * np.log(A)) + ((1 - Y) * np.log(1 - A)))
    dw = (1/m) * np.dot(X, (A - Y).T)
    db = (1/m) * np.sum(A - Y)
    cost = np.squeeze(np.array(cost))

    gradients = {"dw": dw,
                 "db": db}
    return gradients, cost

In [ ]:
def optimize(w, b, X, Y, iterations = 100, learning_rate = 0.009, print_cost = False):
    w = copy.deepcopy(w)
    b = copy.deepcopy(b)
    costs = []

    for i in range(iterations):
        gradients, cost = propagate(w, b, X, Y)
        
        w -= learning_rate * gradients["dw"]
        b -= learning_rate * gradients["db"]

        if i % 100 == 0:
            costs.append(cost)
            if print_cost:
                print(f"Cost after iteration {i}: {cost:.6f}")

    parameters = {"w": w, "b": b}    
    return parameters, gradients, costs

In [ ]:
def predict(w, b, X):
    prediction = np.zeros((1, X.shape[1]))
    w = w.reshape(X.shape[0], 1)

    A = sigmoid(np.dot(w.T, X) + b)

    for i in range(A.shape[1]):
        if A[0, i] > 0.5:
            prediction[0, i] = 1
        else:
            prediction[0, i] = 0
    return prediction         

In [ ]:
def model(X_train, Y_train, X_test, Y_test, iterations=2000, learning_rate=0.5, print_cost=False):
    w, b = initialize(X_train.shape[0])
    
    parameters, gradients, costs = optimize(w, b, X_train, Y_train, iterations, learning_rate, print_cost)
    
    w = parameters["w"]
    b = parameters["b"]
    
    prediction_test = predict(w, b, X_test)
    prediction_train = predict(w, b, X_train)

    if print_cost:
        print(f"Train accuracy: {100 - np.mean(np.abs(prediction_train - Y_train)) * 100} %")
        print(f"Test accuracy: {100 - np.mean(np.abs(prediction_test - Y_test)) * 100} %")

    
    d = {"costs": costs,
         "prediction_test": prediction_test, 
         "prediction_train" : prediction_train, 
         "w" : w, 
         "b" : b,
         "learning_rate" : learning_rate,
         "iterations": iterations}
    
    return d

In [ ]:
logistic_regression_model = model(train_set_x, train_set_y, test_set_x, test_set_y, iterations=2000, learning_rate=0.005, print_cost=True)

In [ ]:
index = 0
plt.imshow(test_set_x[:, index].reshape((num_px, num_px, 3)))
pred = int(logistic_regression_model['prediction_test'][0, index])
actual = int(test_set_y[0, index])
print(f"y = {actual}, the model predicted that it is a '{classes[pred].decode('utf-8')}' picture.")